In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

# 1. Initialize the Model
# It will download the weights the first time you run this (around 1-2 GB)
model_name = "BAAI/bge-large-en-v1.5"
encoder = SentenceTransformer(model_name)

# 2. Prepare your data (Documents and a Query)
documents = [
    "Section 302 of the Indian Penal Code prescribes the punishment for murder.",
    "A writ of Habeas Corpus is used to bring a prisoner or other detainee before the court.",
    "The petition was dismissed by the High Court due to a lack of evidence."
]

# Note: bge-large-en-v1.5 requires a specific prompt prefix ONLY for the search query, not the documents.
query = "Represent this sentence for searching relevant passages: What is the punishment for murder in India?"

# 3. Generate the Embeddings
# This converts your text into dense vector arrays
doc_embeddings = encoder.encode(documents).tolist() 
query_embedding = encoder.encode([query]).tolist()

# 4. Store in ChromaDB
# Initialize a local ChromaDB client
client = chromadb.PersistentClient(path="./legal_chroma_db")
collection = client.get_or_create_collection(name="judgments_trial")

# Add the documents and their generated embeddings to the database
collection.add(
    ids=["doc1", "doc2", "doc3"],
    embeddings=doc_embeddings,
    documents=documents,
    metadatas=[{"case": "A v B"}, {"case": "C v D"}, {"case": "E v F"}] # Your extracted metadata
)

# 5. Query the Database
results = collection.query(
    query_embeddings=query_embedding,
    n_results=1 # Fetch the top 1 most relevant chunk
)

print("Most relevant document:", results['documents'][0])

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6979.53it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Most relevant document: ['Section 302 of the Indian Penal Code prescribes the punishment for murder.']


In [10]:
len(query_embedding[0])

1024

In [8]:
len(doc_embeddings[0])

1024

In [1]:
file_path = "../Docs/2023-odisa-files/display_judgement20pdf.pdf"


from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader(file_path)    
data = loader.load()

/opt/anaconda3/envs/gen_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
data

[Document(metadata={'source': '../Docs/2023-odisa-files/display_judgement20pdf.pdf', 'file_path': '../Docs/2023-odisa-files/display_judgement20pdf.pdf', 'page': 0, 'total_pages': 25, 'Producer': 'GPL Ghostscript 10.01.1', 'CreationDate': "D:20230706110200+05'30'", 'ModDate': "D:20230706110200+05'30'", 'Creator': 'PDF24 Creator'}, page_content='IN THE HIGH COURT OF ORISSA AT CUTTACK\nCRLMC NO.1462 OF 2023\n(Application under Section 482 of the Code of Criminal\nProcedure for fresh investigation or re-investigation by any\nindependent agency)\nBandhna Toppo\n… Petitioner\n-versus-\nState of Orissa\nand others … Opposite Parties\nAdvocates appeared in the case through hybrid mode:\nFor Petitioner: Mr.Shivsankar Mohanty,\nAdvocate\n-versus-\nFor Opp.Parties: Mr.S.N. Das,\nAddl. Standing Counsel\n---------------------------------------------------------------------------\nCORAM:\nJUSTICE SASHIKANTA MISHRA\nJUDGMENT\n05.7.2023.\nCRLMC No.1462 of 2023 Page 1 of 25\n'),
 Document(metadata={'so

In [17]:
with open('display_judgement20pdf.txt', 'w') as f:
    for page in data:
        f.write(page.page_content)
        f.write("\n\n")  # Add a newline between pages

In [2]:
from pydantic import BaseModel, Field
from typing import List, Optional

class JudgmentMetadata(BaseModel):
    court_name: str = Field(description="The name of the court, e.g., High Court of Judicature at Patna, Supreme Court of India.")
    case_number: str = Field(description="The formal case number or writ petition number, e.g., W.P. (C) No. 123 of 2024.")
    judgment_date: str = Field(description="The exact date the judgment was delivered, formatted as YYYY-MM-DD.")
    petitioner: str = Field(description="The name of the petitioner or appellant.")
    respondent: str = Field(description="The name of the respondent.")
    judges: List[str] = Field(description="List of judges who delivered or presided over the judgment.")
    acts_and_sections: List[str] = Field(description="Specific legal acts and sections mentioned in the header, e.g., Section 302 IPC, Article 226.")
    

In [3]:
import os
import json
import pdfplumber
from openai import OpenAI

# Initialize OpenAI client (Ensure your OPENAI_API_KEY environment variable is set)
client = OpenAI()

def extract_raw_text_from_pdf(pdf_path: str, max_pages: int = 3) -> str:
    """Extracts raw text from the first few pages of the judgment PDF."""
    full_text = []
    with pdfplumber.open(pdf_path) as pdf:
        # Loop only through the first few pages to capture the layout/header
        for i, page in enumerate(pdf.pages):
            if i >= max_pages:
                break
            text = page.extract_text()
            if text:
                full_text.append(text)
                
    return "\n--- PAGE BREAK ---\n".join(full_text)

def extract_judgment_metadata(pdf_path: str) -> dict:
    """Parses legal header text using OpenAI Structured Outputs."""
    print(f"Reading file: {os.path.basename(pdf_path)}")
    header_text = extract_raw_text_from_pdf(pdf_path, max_pages=3)
    
    if not header_text.strip():
        raise ValueError(f"Could not extract any text from the PDF: {pdf_path}")

    print("Extracting structured metadata via OpenAI...")
    
    # We use gpt-4o-mini as it is highly accurate for data extraction and incredibly cheap
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system", 
                "content": "You are an expert legal clerk specialized in parsing Indian High Court and State Court judgment headers. Extract the core metadata accurately from the provided text snippet."
            },
            {
                "role": "user", 
                "content": f"Extract metadata from this Indian Court judgment text:\n\n{header_text}"
            }
        ],
        response_format=JudgmentMetadata, # Enforces the exact Pydantic schema structure
        temperature=0.0
    )
    
    # Return parsed data as a Python dictionary
    return response.choices[0].message.parsed.model_dump()

# --- Execution Example ---
if __name__ == "__main__":
    # Replace with an actual path to an Indian court judgment PDF on your Mac
    # sample_pdf = "path/to/your/patna_high_court_judgment.pdf" 
    sample_pdf = file_path
    
    try:
        metadata = extract_judgment_metadata(sample_pdf)
        print("\nSuccessfully Extracted Metadata JSON:")
        print(json.dumps(metadata, indent=4))
    except Exception as e:
        print(f"Error occurred: {e}")

Reading file: display_judgement20pdf.pdf
Extracting structured metadata via OpenAI...

Successfully Extracted Metadata JSON:
{
    "court_name": "High Court of Orissa at Cuttack",
    "case_number": "CRLMC NO.1462 OF 2023",
    "judgment_date": "2023-07-05",
    "petitioner": "Bandhna Toppo",
    "respondent": "State of Orissa and others",
    "judges": [
        "Justice Sashikanta Mishra"
    ],
    "acts_and_sections": [
        "Section 482 of the Code of Criminal Procedure",
        "Section 302 IPC",
        "Section 34 IPC"
    ]
}


# Connection Testing of remote Chroma DB

In [3]:
import chromadb

# Replace with your Chroma server details
# HOST = "manishs-mac-mini.tailc96719.ts.net"
HOST="100.97.38.95"
PORT = 8030

client = chromadb.HttpClient(
    host=HOST,
    port=PORT,
)

try:
    collections = client.list_collections()
    print("✅ Connected successfully!")
    print(f"Collections: {collections}")
except Exception as e:
    print("❌ Connection failed")
    print(e)

✅ Connected successfully!
Collections: []


In [ ]:
curl http://100.97.38.95:8030/api/v2/heartbeat